# 05 SHAP解释（任务书：SHAP、分区主控因子变化）

In [ ]:
import sys
from pathlib import Path
# 让 notebook 能 import src
sys.path.append(str(Path('..').resolve()))

import pandas as pd
import numpy as np

from src.utils import load_config, ensure_dir, set_seed

cfg = load_config('../config/config.yaml')
OUT = ensure_dir('../outputs')
SHAP_DIR = ensure_dir(OUT/'shap')
df = pd.read_parquet(OUT/'bcf_final.parquet')


In [2]:
from src.spatial_cv import make_spatial_groups, get_group_kfold
from src.models import fit_oof_with_spatialcv
from src.explain_shap import compute_shap_for_pipeline, save_shap_summary
from src.utils import safe_col
import numpy as np

bcf_col = safe_col(df, cfg['data']['columns']['target_bcf'])
cv = get_group_kfold(cfg)

preferred = [m for m in ['xgb','rf'] if m in cfg['modeling']['models']]
if not preferred:
    raise RuntimeError("请在config里至少包含 xgb 或 rf 以高效做SHAP")
m = preferred[0]

group_keys = ['crop','ph_bin'] if 'crop' in df.columns else ['ph_bin']
summary_rows = []
for key, sub in df.groupby(group_keys, dropna=False):
    sub = sub.reset_index(drop=True)
    gsub = make_spatial_groups(sub, cfg)
    res = fit_oof_with_spatialcv(sub, cfg, gsub, m, cv)

    X = sub.drop(columns=[bcf_col], errors='ignore')
    sv, feat_names = compute_shap_for_pipeline(res.best_estimator, X, max_rows=int(cfg['shap']['sample_rows']), random_state=cfg['project']['seed'])

    key_str = "_".join(map(str, key if isinstance(key, tuple) else (key,)))
    save_shap_summary(sv, feat_names, str(SHAP_DIR/f"shap_summary_{key_str}_{m}.png"))

    mean_abs = np.mean(np.abs(sv), axis=0)
    topk = np.argsort(mean_abs)[::-1][:int(cfg['shap']['top_k'])]
    for i in topk:
        summary_rows.append({"group": str(key), "model_for_shap": m, "feature": feat_names[i], "mean_abs_shap": float(mean_abs[i])})

rank = pd.DataFrame(summary_rows)
rank.to_excel(OUT/'shap_rank_compare.xlsx', index=False)
rank.head(20)


c:\My File\cao\soil\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\My File\cao\soil\soil_task_project\src\explain_shap.py:23: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_values, features=None, feature_names=feature_names, show=False)
C:\My File\cao\soil\soil_task_project\src\explain_shap.py:23: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_values, features=None, feature_names=feature_names, show=False)


,group,model_for_shap,feature,mean_abs_shap
0,"('corn', 'acid')",xgb,soil Cd,0.022370
1,"('corn', 'acid')",xgb,crop Cd,0.022360
2,"('corn', 'acid')",xgb,water content,0.000432
3,"('corn', 'acid')",xgb,geology,0.000332
4,"('corn', 'acid')",xgb,pH,0.000281
5,"('corn', 'acid')",xgb,Mn,0.000151
6,"('corn', 'acid')",xgb,N,0.000118
7,"('corn', 'acid')",xgb,P,0.000114
8,"('corn', 'acid')",xgb,K,0.000111
9,"('corn', 'acid')",xgb,Distance from pollution source,0.000100
